# GridToEV modelling dataset

This notebook builds one clean, half-hourly modelling table for Ireland by combining:

- Hack the Climate / ENTSO-E generation, load, and price samples;
- EirGrid quarter-hourly system conditions; and
- EirGrid half-hourly renewable dispatch-down labels.

The table is designed for **30- and 60-minute-ahead forecasting**. Every row records an
`issue_timestamp_utc`, a later `target_timestamp_utc`, features known at the issue time,
and targets measured only at the future target time. This explicit separation reduces the
risk of accidentally training on the value being predicted.

Raw source files remain unchanged under `data/raw/`. The final output is written to
`data/processed/gridtoev_model_ready.csv`.

## Modelling assumptions

- The organiser sample timestamps are interpreted as UTC. During the January overlap used
  here, Irish local time is also UTC, so daylight-saving ambiguity does not affect the result.
- EirGrid timestamps are converted to UTC using each file's `GMT Offset` field.
- Prices for the Irish SEM sample are hourly. They are carried forward once to match the
  30-minute modelling grid; a flag identifies the carried-forward rows.
- A short gap of at most one hour in an observed feature may be time-interpolated and is
  flagged. Longer gaps are not fabricated; affected modelling rows are removed.
- `recoverable_surplus_100mw_flex_mwh` is a transparent scenario metric for a 100 MW flexible
  load operating for 30 minutes. It is not a claim that this flexibility existed historically.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import re
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_DATASET = PROCESSED_DIR / "gridtoev_model_ready.csv"
OUTPUT_DICTIONARY = PROCESSED_DIR / "gridtoev_data_dictionary.csv"
OUTPUT_QUALITY = PROCESSED_DIR / "gridtoev_quality_report.json"
OUTPUT_MANIFEST = PROCESSED_DIR / "source_manifest.csv"

FREQUENCY = "30min"
FORECAST_HORIZONS_MINUTES = (30, 60)
FLEXIBLE_LOAD_CAPACITY_MW = 100.0
INTERVAL_HOURS = 0.5

## 1. Source acquisition and provenance

Downloads are idempotent: an existing raw file is never overwritten. This makes the notebook
suitable for an offline hackathon demo after the first successful run. The SHA-256 manifest
records the exact bytes used to build the processed dataset.

In [2]:
SOURCES = {
    "generation": {
        "url": "https://hacktheclimate.io/samples/generation.csv",
        "path": RAW_DIR / "hacktheclimate" / "generation.csv",
    },
    "load": {
        "url": "https://hacktheclimate.io/samples/load.csv",
        "path": RAW_DIR / "hacktheclimate" / "load.csv",
    },
    "prices": {
        "url": "https://hacktheclimate.io/samples/prices.csv",
        "path": RAW_DIR / "hacktheclimate" / "prices.csv",
    },
    "eirgrid_system_2026": {
        "url": "https://cms.eirgrid.ie/sites/default/files/publications/System-Data-Qtr-Hourly-2026-V7.xlsx",
        "path": RAW_DIR / "eirgrid" / "system_data_qtr_hourly_2026_v7.xlsx",
    },
    "eirgrid_dispatch_down_2026": {
        "url": "https://cms.eirgrid.ie/sites/default/files/publications/DD-HH-2026-V9.xlsx",
        "path": RAW_DIR / "eirgrid" / "dd_half_hourly_2026_v9.xlsx",
    },
}


def download_if_missing(url: str, destination: Path) -> None:
    # Download a public source once while leaving an existing raw file untouched.
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return
    request = urllib.request.Request(url, headers={"User-Agent": "GridToEV/1.0"})
    with urllib.request.urlopen(request, timeout=120) as response:
        destination.write_bytes(response.read())


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


for source in SOURCES.values():
    download_if_missing(source["url"], source["path"])

manifest = pd.DataFrame(
    [
        {
            "source": name,
            "url": source["url"],
            "local_path": source["path"].relative_to(REPO_ROOT).as_posix(),
            "bytes": source["path"].stat().st_size,
            "sha256": sha256(source["path"]),
        }
        for name, source in SOURCES.items()
    ]
)
manifest.to_csv(OUTPUT_MANIFEST, index=False)
display(manifest)

,source,url,local_path,bytes,sha256
0,generation,https://hacktheclimate.io/samples/generation.csv,data/raw/hacktheclimate/generation.csv,839000,89b7dbb42126dcbcb87d1b9c1213daea2d548c7a1a3246...
1,load,https://hacktheclimate.io/samples/load.csv,data/raw/hacktheclimate/load.csv,92890,c6e06553d9101a4fc7c14a7dfdcba78ad22cb3057f2e12...
2,prices,https://hacktheclimate.io/samples/prices.csv,data/raw/hacktheclimate/prices.csv,116993,ad19188ed88a2134249c030b56e45b0cd8dce646cb5d5a...
3,eirgrid_system_2026,https://cms.eirgrid.ie/sites/default/files/pub...,data/raw/eirgrid/system_data_qtr_hourly_2026_v...,7270788,7f2f046dc04bcd43f4bf30c065d9f383c99e8f15c7c3f8...
4,eirgrid_dispatch_down_2026,https://cms.eirgrid.ie/sites/default/files/pub...,data/raw/eirgrid/dd_half_hourly_2026_v9.xlsx,3417425,b641876ea690ca5c4ffc180b8ba99cac80518ee164caa0...


## 2. Clean organiser generation, load, and price samples

Generation is pivoted from production-type rows to one row per Irish half-hour. Source names
stay visible in the column names. Missing timestamps are interpolated only across gaps of no
more than two half-hour steps, and the notebook retains imputation flags.

In [3]:
def snake_case(value: str) -> str:
    value = re.sub(r"[^a-zA-Z0-9]+", "_", value.strip()).strip("_")
    return value.lower()


generation_raw = pd.read_csv(SOURCES["generation"]["path"])
generation_raw["timestamp_utc"] = pd.to_datetime(
    generation_raw["DateTime"], errors="raise", utc=True
)
generation_ie = generation_raw.loc[generation_raw["MapCode"].eq("IE")].copy()

generation = generation_ie.pivot_table(
    index="timestamp_utc",
    columns="ProductionType",
    values="ActualGenerationOutput",
    aggfunc="sum",
).sort_index()
generation.columns = [
    f"entsoe_{snake_case(column)}_generation_mw" for column in generation.columns
]

generation_index = pd.date_range(
    generation.index.min(), generation.index.max(), freq=FREQUENCY, tz="UTC"
)
generation = generation.reindex(generation_index)
generation_missing_before_fill = generation.isna().any(axis=1)
generation = generation.interpolate(method="time", limit=2, limit_area="inside")
generation["entsoe_generation_imputed_flag"] = generation_missing_before_fill.astype("int8")

renewable_generation_columns = [
    column
    for column in generation.columns
    if any(token in column for token in ("wind_", "solar_", "hydro_run_of_river"))
]
numeric_generation_columns = [
    column for column in generation.columns if column.endswith("_generation_mw")
]
generation["entsoe_total_generation_mw"] = generation[numeric_generation_columns].sum(axis=1)
generation["entsoe_renewable_generation_mw"] = generation[
    renewable_generation_columns
].sum(axis=1)
generation["entsoe_nonrenewable_generation_mw"] = (
    generation["entsoe_total_generation_mw"]
    - generation["entsoe_renewable_generation_mw"]
)

load_raw = pd.read_csv(SOURCES["load"]["path"])
load_raw["timestamp_utc"] = pd.to_datetime(load_raw["DateTime"], errors="raise", utc=True)
load = (
    load_raw.loc[load_raw["MapCode"].eq("IE"), ["timestamp_utc", "TotalLoadValue"]]
    .drop_duplicates("timestamp_utc", keep="last")
    .set_index("timestamp_utc")
    .sort_index()
    .rename(columns={"TotalLoadValue": "entsoe_actual_load_mw"})
)
load_index = pd.date_range(load.index.min(), load.index.max(), freq=FREQUENCY, tz="UTC")
load = load.reindex(load_index)
load_missing_before_fill = load["entsoe_actual_load_mw"].isna()
load["entsoe_actual_load_mw"] = load["entsoe_actual_load_mw"].interpolate(
    method="time", limit=2, limit_area="inside"
)
load["entsoe_load_imputed_flag"] = load_missing_before_fill.astype("int8")

prices_raw = pd.read_csv(SOURCES["prices"]["path"])
prices_raw["timestamp_utc"] = pd.to_datetime(prices_raw["DateTime"], errors="raise", utc=True)
prices = (
    prices_raw.loc[
        prices_raw["MapCode"].eq("IE_SEM"), ["timestamp_utc", "Price[Currency/MWh]"]
    ]
    .drop_duplicates("timestamp_utc", keep="last")
    .set_index("timestamp_utc")
    .sort_index()
    .rename(columns={"Price[Currency/MWh]": "entsoe_price_eur_mwh"})
    .resample(FREQUENCY)
    .ffill(limit=1)
)
prices["entsoe_price_hourly_carry_forward_flag"] = (
    prices.index.minute == 30
).astype("int8")

print("Generation rows:", len(generation))
print("Load rows:", len(load))
print("Price rows after half-hour alignment:", len(prices))
display(generation.head(3))

Generation rows: 1488
Load rows: 1488
Price rows after half-hour alignment: 1487


,entsoe_fossil_gas_generation_mw,entsoe_fossil_hard_coal_generation_mw,entsoe_fossil_oil_generation_mw,entsoe_fossil_peat_generation_mw,entsoe_hydro_pumped_storage_generation_mw,entsoe_hydro_run_of_river_and_poundage_generation_mw,entsoe_other_generation_mw,entsoe_solar_generation_mw,entsoe_wind_onshore_generation_mw,entsoe_generation_imputed_flag,entsoe_total_generation_mw,entsoe_renewable_generation_mw,entsoe_nonrenewable_generation_mw
2026-01-01 00:00:00+00:00,639.810,0.0,167.717,41.202,0.0,63.620,0.0,0.0,1326.084,0,2238.433,1389.704,848.729
2026-01-01 00:30:00+00:00,604.560,0.0,173.771,41.061,0.0,63.784,0.0,0.0,1416.327,0,2299.503,1480.111,819.392
2026-01-01 01:00:00+00:00,705.741,0.0,166.042,41.339,0.0,63.626,0.0,0.0,1404.501,0,2381.249,1468.127,913.122


## 3. Clean EirGrid system conditions

EirGrid's 15-minute SCADA averages are resampled to half-hour means. Both Ireland-only and
all-island fields are retained because dispatch-down can arise from local constraints or
wider system conditions. `SNSP` is already supplied by EirGrid, so it is preferred to a
reconstructed proxy.

In [4]:
SYSTEM_RENAME = {
    "IE Generation": "eirgrid_ie_generation_mw",
    "IE Demand": "eirgrid_ie_demand_mw",
    "IE Wind Availability": "eirgrid_ie_wind_availability_mw",
    "IE Wind Generation": "eirgrid_ie_wind_generation_mw",
    "IE Solar Availability": "eirgrid_ie_solar_availability_mw",
    "IE Solar Generation": "eirgrid_ie_solar_generation_mw",
    "IE Hydro": "eirgrid_ie_hydro_generation_mw",
    "EWIC I/C": "eirgrid_ewic_flow_mw",
    "Greenlink I/C": "eirgrid_greenlink_flow_mw",
    "IE Wind Penetration": "eirgrid_ie_wind_penetration_ratio",
    "IE Solar Penetration": "eirgrid_ie_solar_penetration_ratio",
    "AI Generation": "eirgrid_all_island_generation_mw",
    "AI Demand": "eirgrid_all_island_demand_mw",
    "AI Wind Availability": "eirgrid_all_island_wind_availability_mw",
    "AI Wind Generation": "eirgrid_all_island_wind_generation_mw",
    "AI Solar Availability": "eirgrid_all_island_solar_availability_mw",
    "AI Solar Generation": "eirgrid_all_island_solar_generation_mw",
    "AI Hydro": "eirgrid_all_island_hydro_generation_mw",
    "Inter-Jurisdictional Flow": "eirgrid_interjurisdictional_flow_mw",
    "AI Wind Penetration": "eirgrid_all_island_wind_penetration_ratio",
    "AI Solar Penetration": "eirgrid_all_island_solar_penetration_ratio",
    "AI Oversupply": "eirgrid_all_island_oversupply_mw",
    "AI Oversupply Percentage": "eirgrid_all_island_oversupply_ratio",
    "SNSP": "eirgrid_snsp_ratio",
}

system_raw = pd.read_excel(SOURCES["eirgrid_system_2026"]["path"], sheet_name="System Data")
system_raw["timestamp_utc"] = (
    pd.to_datetime(system_raw["DateTime"], errors="raise")
    - pd.to_timedelta(system_raw["GMT Offset"], unit="h")
).dt.tz_localize("UTC")
system = (
    system_raw.set_index("timestamp_utc")[list(SYSTEM_RENAME)]
    .rename(columns=SYSTEM_RENAME)
    .apply(pd.to_numeric, errors="coerce")
    .sort_index()
    .resample(FREQUENCY)
    .mean()
)
system_missing_before_fill = system.isna().any(axis=1)
system = system.interpolate(method="time", limit=2, limit_area="inside")
system["eirgrid_system_imputed_flag"] = system_missing_before_fill.astype("int8")

print("System coverage:", system.index.min(), "to", system.index.max())
display(system.head(3))

System coverage: 2026-01-01 00:00:00+00:00 to 2026-07-31 22:30:00+00:00


,eirgrid_ie_generation_mw,eirgrid_ie_demand_mw,eirgrid_ie_wind_availability_mw,eirgrid_ie_wind_generation_mw,eirgrid_ie_solar_availability_mw,eirgrid_ie_solar_generation_mw,eirgrid_ie_hydro_generation_mw,eirgrid_ewic_flow_mw,eirgrid_greenlink_flow_mw,eirgrid_ie_wind_penetration_ratio,eirgrid_ie_solar_penetration_ratio,eirgrid_all_island_generation_mw,eirgrid_all_island_demand_mw,eirgrid_all_island_wind_availability_mw,eirgrid_all_island_wind_generation_mw,eirgrid_all_island_solar_availability_mw,eirgrid_all_island_solar_generation_mw,eirgrid_all_island_hydro_generation_mw,eirgrid_interjurisdictional_flow_mw,eirgrid_all_island_wind_penetration_ratio,eirgrid_all_island_solar_penetration_ratio,eirgrid_all_island_oversupply_mw,eirgrid_all_island_oversupply_ratio,eirgrid_snsp_ratio,eirgrid_system_imputed_flag
timestamp_utc,,,,,,,,,,,,,,,,,,,,,,,,,
2026-01-01 00:00:00+00:00,2566.155,3893.295,1402.620,1370.85,0.580,4.210,63.550,524.030,513.190,0.352117,0.001081,3156.9205,4495.3380,2076.955,1547.3545,2.159,4.210,63.550,-311.390,0.344228,0.000937,0.0,0.0,0.6409,0
2026-01-01 00:30:00+00:00,2667.045,3832.990,1443.130,1396.20,0.590,4.230,63.775,451.015,513.205,0.364270,0.001104,3256.2970,4408.2655,2120.599,1572.5030,2.169,4.230,63.775,-351.175,0.356732,0.000960,0.0,0.0,0.6274,0
2026-01-01 01:00:00+00:00,2619.155,3758.925,1419.125,1356.86,0.575,4.215,63.725,524.430,513.238,0.360943,0.001121,3216.6390,4353.9895,2088.352,1554.6820,2.154,4.215,63.725,-347.415,0.357072,0.000968,0.0,0.0,0.6339,0


## 4. Build dispatch-down labels from the EirGrid ground truth

EirGrid supplies MWh per half-hour separately for wind and solar. The notebook filters to
Ireland and sums those renewable types. Curtailment, network constraints, and other
reductions remain separate, while total dispatch-down is retained as the main regression
label.

In [5]:
DD_RENAME = {
    "Sum of AV_MWH": "available_renewable_mwh_observed",
    "Sum of AO_MWH": "actual_renewable_output_mwh_observed",
    "Sum of HI_FRQ_MIN_GEN_MWH": "high_frequency_min_generation_mwh_observed",
    "Sum of ROCOF_INERTIA_MWH": "rocof_inertia_mwh_observed",
    "Sum of SNSP_MWH": "snsp_curtailment_mwh_observed",
    "Sum of TRANS_CONSTR_MWH": "transmission_constraint_mwh_observed",
    "Sum of TSO_TEST_MWH": "tso_test_mwh_observed",
    "Sum of DD_MWH": "dispatch_down_mwh_observed",
    "Sum of CURTAILMENTS_MWH": "curtailment_mwh_observed",
    "Sum of CONSTRAINTS_MWH": "constraint_mwh_observed",
    "Sum of OTHER_MWH": "other_reduction_mwh_observed",
}

dispatch_raw = pd.read_excel(
    SOURCES["eirgrid_dispatch_down_2026"]["path"], sheet_name="DD HH"
)
dispatch_ie = dispatch_raw.loc[dispatch_raw["JURISDICTION"].eq("IE")].copy()
dispatch_ie["timestamp_utc"] = (
    pd.to_datetime(dispatch_ie["HH_TIMESTAMP"], errors="raise")
    - pd.to_timedelta(dispatch_ie["GMT_OFFSET"], unit="h")
).dt.tz_localize("UTC")

dispatch = (
    dispatch_ie.groupby("timestamp_utc", as_index=True)[list(DD_RENAME)]
    .sum(min_count=1)
    .rename(columns=DD_RENAME)
    .apply(pd.to_numeric, errors="coerce")
    .sort_index()
)

# EirGrid's total can differ by tiny rounding amounts, so validate with a small tolerance.
reconstructed_total = (
    dispatch["curtailment_mwh_observed"]
    + dispatch["constraint_mwh_observed"]
)
max_accounting_difference = (
    dispatch["dispatch_down_mwh_observed"] - reconstructed_total
).abs().max()
print("Dispatch-down coverage:", dispatch.index.min(), "to", dispatch.index.max())
print("Maximum target accounting difference (MWh):", round(max_accounting_difference, 6))
display(dispatch.head(3))

Dispatch-down coverage: 2026-01-01 00:00:00+00:00 to 2026-08-31 22:30:00+00:00
Maximum target accounting difference (MWh): 0.031


,available_renewable_mwh_observed,actual_renewable_output_mwh_observed,high_frequency_min_generation_mwh_observed,rocof_inertia_mwh_observed,snsp_curtailment_mwh_observed,transmission_constraint_mwh_observed,tso_test_mwh_observed,dispatch_down_mwh_observed,curtailment_mwh_observed,constraint_mwh_observed,other_reduction_mwh_observed
timestamp_utc,,,,,,,,,,,
2026-01-01 00:00:00+00:00,688.870,688.213,0.0,0.0,0.0,0.492,0.0,0.492,0.0,0.492,0.0
2026-01-01 00:30:00+00:00,707.783,700.758,0.0,0.0,0.0,5.565,0.0,5.565,0.0,5.565,0.0
2026-01-01 01:00:00+00:00,687.931,681.113,0.0,0.0,0.0,4.550,0.0,4.550,0.0,4.550,0.0


## 5. Merge the source-aligned feature table

The shared modelling window is the overlap of all organiser samples and EirGrid data. This
deliberately favours a fully populated multi-source table over a longer table with fabricated
market prices. Extreme and negative values are preserved because they may describe genuine
power-system conditions.

In [6]:
overlap_start = max(
    frame.index.min() for frame in (generation, load, prices, system, dispatch)
)
overlap_end = min(
    frame.index.max() for frame in (generation, load, prices, system, dispatch)
)
modelling_index = pd.date_range(overlap_start, overlap_end, freq=FREQUENCY, tz="UTC")

base = pd.concat(
    [
        generation.reindex(modelling_index),
        load.reindex(modelling_index),
        prices.reindex(modelling_index),
        system.reindex(modelling_index),
        dispatch.reindex(modelling_index),
    ],
    axis=1,
)
base.index.name = "issue_timestamp_utc"

# Physically interpretable state features.
base["eirgrid_ie_variable_renewable_generation_mw"] = (
    base["eirgrid_ie_wind_generation_mw"] + base["eirgrid_ie_solar_generation_mw"]
)
base["eirgrid_ie_variable_renewable_availability_mw"] = (
    base["eirgrid_ie_wind_availability_mw"] + base["eirgrid_ie_solar_availability_mw"]
)
base["eirgrid_ie_renewable_headroom_mw"] = (
    base["eirgrid_ie_variable_renewable_availability_mw"]
    - base["eirgrid_ie_variable_renewable_generation_mw"]
).clip(lower=0)
base["eirgrid_ie_net_load_mw"] = (
    base["eirgrid_ie_demand_mw"]
    - base["eirgrid_ie_variable_renewable_generation_mw"]
)
base["eirgrid_ie_renewable_share_ratio"] = (
    base["eirgrid_ie_variable_renewable_generation_mw"]
    / base["eirgrid_ie_demand_mw"].replace(0, np.nan)
)
base["eirgrid_snsp_headroom_to_75pct"] = 0.75 - base["eirgrid_snsp_ratio"]
base["entsoe_price_negative_flag"] = (base["entsoe_price_eur_mwh"] < 0).astype("int8")
base["entsoe_price_below_20_flag"] = (base["entsoe_price_eur_mwh"] < 20).astype("int8")
base["high_wind_low_load_interaction"] = (
    base["eirgrid_ie_wind_generation_mw"]
    / base["eirgrid_ie_demand_mw"].replace(0, np.nan)
)

# Cyclical calendar features avoid artificial jumps between 23:30 and 00:00.
half_hour_slot = base.index.hour * 2 + (base.index.minute // 30)
base["hour_sin"] = np.sin(2 * np.pi * half_hour_slot / 48)
base["hour_cos"] = np.cos(2 * np.pi * half_hour_slot / 48)
base["weekday"] = base.index.dayofweek.astype("int8")
base["weekday_sin"] = np.sin(2 * np.pi * base["weekday"] / 7)
base["weekday_cos"] = np.cos(2 * np.pi * base["weekday"] / 7)
base["is_weekend"] = (base["weekday"] >= 5).astype("int8")
base["month"] = base.index.month.astype("int8")

print("Common source overlap:", overlap_start, "to", overlap_end)
print("Half-hour rows before lag/target trimming:", len(base))

Common source overlap: 2026-01-01 00:00:00+00:00 to 2026-01-31 23:00:00+00:00
Half-hour rows before lag/target trimming: 1487


## 6. Engineer causal latest-state, lags, ramps, and rolling features

The latest completed dispatch-down interval is exposed as an issue-time feature. Its timestamp
is the issue timestamp, while every label is shifted to a strictly later target timestamp, so
it does not reveal the future value being predicted. Older lags remain available for trend
estimation. Rolling features use `.shift(1)` so their history ends before the issue time.

In [7]:
LAG_SOURCES = {
    "wind": "eirgrid_ie_wind_generation_mw",
    "load": "eirgrid_ie_demand_mw",
    "price": "entsoe_price_eur_mwh",
    "snsp": "eirgrid_snsp_ratio",
    "oversupply": "eirgrid_all_island_oversupply_mw",
}
LAG_STEPS = (1, 2, 4, 12, 48)  # 30m, 1h, 2h, 6h, 24h

# These are observations for the completed interval available at issue time, not future labels.
base["dispatch_down_mwh_latest_observed"] = base["dispatch_down_mwh_observed"]
base["dispatch_down_event_latest_observed"] = (
    base["dispatch_down_mwh_observed"] > 0
).astype("int8")

for short_name, source_column in LAG_SOURCES.items():
    for lag in LAG_STEPS:
        base[f"{short_name}_lag_{lag}"] = base[source_column].shift(lag)
    base[f"{short_name}_ramp_30m"] = base[source_column].diff(1)
    base[f"{short_name}_ramp_60m"] = base[source_column].diff(2)

for lag in (1, 2, 4, 48):
    base[f"dispatch_down_mwh_lag_{lag}"] = base[
        "dispatch_down_mwh_observed"
    ].shift(lag)
    base[f"dispatch_down_event_lag_{lag}"] = (
        base["dispatch_down_mwh_observed"].shift(lag) > 0
    ).astype("int8")

for window in (6, 12, 48):  # 3h, 6h, 24h on a half-hour grid
    historical_wind = base["eirgrid_ie_wind_generation_mw"].shift(1)
    historical_load = base["eirgrid_ie_demand_mw"].shift(1)
    historical_price = base["entsoe_price_eur_mwh"].shift(1)
    base[f"wind_rolling_mean_{window}"] = historical_wind.rolling(window).mean()
    base[f"wind_rolling_std_{window}"] = historical_wind.rolling(window).std()
    base[f"load_rolling_mean_{window}"] = historical_load.rolling(window).mean()
    base[f"price_rolling_mean_{window}"] = historical_price.rolling(window).mean()
    base[f"price_rolling_std_{window}"] = historical_price.rolling(window).std()

## 7. Attach future targets for 30- and 60-minute horizons

The output uses a long format: each issue time appears once per forecast horizon. This lets a
single estimator use `forecast_horizon_minutes` as a feature or lets separate horizon-specific
models filter the same table.

In [8]:
OBSERVED_LABEL_COLUMNS = list(DD_RENAME.values())
TARGET_RENAME = {
    "high_frequency_min_generation_mwh_observed": "high_frequency_min_generation_mwh",
    "rocof_inertia_mwh_observed": "rocof_inertia_mwh",
    "snsp_curtailment_mwh_observed": "snsp_curtailment_mwh",
    "transmission_constraint_mwh_observed": "transmission_constraint_mwh",
    "tso_test_mwh_observed": "tso_test_mwh",
    "dispatch_down_mwh_observed": "dispatch_down_mwh",
    "curtailment_mwh_observed": "curtailment_mwh",
    "constraint_mwh_observed": "constraint_mwh",
    "other_reduction_mwh_observed": "other_reduction_mwh",
}

feature_columns = [
    column for column in base.columns if column not in OBSERVED_LABEL_COLUMNS
]
horizon_blocks = []

for horizon_minutes in FORECAST_HORIZONS_MINUTES:
    steps = horizon_minutes // 30
    block = base[feature_columns].copy()
    future_targets = base[list(TARGET_RENAME)].shift(-steps).rename(columns=TARGET_RENAME)
    block = block.join(future_targets)
    block["forecast_horizon_minutes"] = horizon_minutes
    block["target_timestamp_utc"] = block.index + pd.Timedelta(minutes=horizon_minutes)
    block["dispatch_down_event"] = (block["dispatch_down_mwh"] > 0).astype("int8")
    block["recoverable_surplus_upper_bound_mwh"] = block["dispatch_down_mwh"]
    block["recoverable_surplus_100mw_flex_mwh"] = np.minimum(
        block["dispatch_down_mwh"],
        FLEXIBLE_LOAD_CAPACITY_MW * INTERVAL_HOURS,
    )
    horizon_blocks.append(block.reset_index())

combined = pd.concat(horizon_blocks, ignore_index=True)
combined = combined.replace([np.inf, -np.inf], np.nan)

target_columns = list(TARGET_RENAME.values()) + [
    "dispatch_down_event",
    "recoverable_surplus_upper_bound_mwh",
    "recoverable_surplus_100mw_flex_mwh",
]
required_columns = feature_columns + target_columns
combined = combined.dropna(subset=required_columns)
combined = combined.sort_values(
    ["issue_timestamp_utc", "forecast_horizon_minutes"]
).reset_index(drop=True)

# Use compact integer types for flags after all filtering is complete.
for column in combined.columns:
    if column.endswith("_flag") or column.startswith("is_") or column == "dispatch_down_event":
        combined[column] = combined[column].astype("int8")

print("Final shape:", combined.shape)
display(combined.head(4))

Final shape: (2867, 133)


,issue_timestamp_utc,entsoe_fossil_gas_generation_mw,entsoe_fossil_hard_coal_generation_mw,entsoe_fossil_oil_generation_mw,entsoe_fossil_peat_generation_mw,entsoe_hydro_pumped_storage_generation_mw,entsoe_hydro_run_of_river_and_poundage_generation_mw,entsoe_other_generation_mw,entsoe_solar_generation_mw,entsoe_wind_onshore_generation_mw,entsoe_generation_imputed_flag,entsoe_total_generation_mw,entsoe_renewable_generation_mw,entsoe_nonrenewable_generation_mw,entsoe_actual_load_mw,entsoe_load_imputed_flag,entsoe_price_eur_mwh,entsoe_price_hourly_carry_forward_flag,eirgrid_ie_generation_mw,eirgrid_ie_demand_mw,eirgrid_ie_wind_availability_mw,eirgrid_ie_wind_generation_mw,eirgrid_ie_solar_availability_mw,eirgrid_ie_solar_generation_mw,eirgrid_ie_hydro_generation_mw,eirgrid_ewic_flow_mw,eirgrid_greenlink_flow_mw,eirgrid_ie_wind_penetration_ratio,eirgrid_ie_solar_penetration_ratio,eirgrid_all_island_generation_mw,eirgrid_all_island_demand_mw,eirgrid_all_island_wind_availability_mw,eirgrid_all_island_wind_generation_mw,eirgrid_all_island_solar_availability_mw,eirgrid_all_island_solar_generation_mw,eirgrid_all_island_hydro_generation_mw,eirgrid_interjurisdictional_flow_mw,eirgrid_all_island_wind_penetration_ratio,eirgrid_all_island_solar_penetration_ratio,eirgrid_all_island_oversupply_mw,eirgrid_all_island_oversupply_ratio,eirgrid_snsp_ratio,eirgrid_system_imputed_flag,eirgrid_ie_variable_renewable_generation_mw,eirgrid_ie_variable_renewable_availability_mw,eirgrid_ie_renewable_headroom_mw,eirgrid_ie_net_load_mw,eirgrid_ie_renewable_share_ratio,eirgrid_snsp_headroom_to_75pct,entsoe_price_negative_flag,entsoe_price_below_20_flag,high_wind_low_load_interaction,hour_sin,hour_cos,weekday,weekday_sin,weekday_cos,is_weekend,month,dispatch_down_mwh_latest_observed,...,load_ramp_30m,load_ramp_60m,price_lag_1,price_lag_2,price_lag_4,price_lag_12,price_lag_48,price_ramp_30m,price_ramp_60m,snsp_lag_1,snsp_lag_2,snsp_lag_4,snsp_lag_12,snsp_lag_48,snsp_ramp_30m,snsp_ramp_60m,oversupply_lag_1,oversupply_lag_2,oversupply_lag_4,oversupply_lag_12,oversupply_lag_48,oversupply_ramp_30m,oversupply_ramp_60m,dispatch_down_mwh_lag_1,dispatch_down_event_lag_1,dispatch_down_mwh_lag_2,dispatch_down_event_lag_2,dispatch_down_mwh_lag_4,dispatch_down_event_lag_4,dispatch_down_mwh_lag_48,dispatch_down_event_lag_48,wind_rolling_mean_6,wind_rolling_std_6,load_rolling_mean_6,price_rolling_mean_6,price_rolling_std_6,wind_rolling_mean_12,wind_rolling_std_12,load_rolling_mean_12,price_rolling_mean_12,price_rolling_std_12,wind_rolling_mean_48,wind_rolling_std_48,load_rolling_mean_48,price_rolling_mean_48,price_rolling_std_48,high_frequency_min_generation_mwh,rocof_inertia_mwh,snsp_curtailment_mwh,transmission_constraint_mwh,tso_test_mwh,dispatch_down_mwh,curtailment_mwh,constraint_mwh,other_reduction_mwh,forecast_horizon_minutes,target_timestamp_utc,dispatch_down_event,recoverable_surplus_upper_bound_mwh,recoverable_surplus_100mw_flex_mwh
0,2026-01-02 00:00:00+00:00,526.529,0.0,0.006,41.339,0.0,63.584,0.0,0.0,1445.882,0,2077.340,1509.466,567.874,3755.400,0,83.0,0,2503.790,3738.43,1710.38,1564.035,0.525,4.13,63.405,529.920,513.1740,0.418416,0.001105,3106.920,4364.7665,2036.659,1740.5595,1.567,4.13,63.405,-391.225,0.398838,0.000946,0.0,0.0,0.69405,0,1568.165,1710.905,142.740,2170.265,0.419472,0.05595,0,0,0.418367,0.000000,1.000000,4,-0.433884,-0.900969,0,1,74.020,...,-82.155,-181.820,84.49,84.49,85.0,169.8,80.0,-1.49,-1.49,0.69245,0.69755,0.69985,0.58955,0.6409,0.0016,-0.0035,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117.536,1,75.015,1,25.959,1,0.492,1,1552.9175,64.777994,4013.497500,93.728333,13.918789,1418.82375,149.905705,4278.392917,120.844167,32.006831,1270.224479,189.449014,3957.865104,99.359792,30.767389,19.999,0.0,0.0,64.736,0.0,84.734,19.999,64.736,0.0,30,2026-01-02 00:30:00+00:00,1,84.734,50.0
1,2026-01-02 00:00:00+00:00,526.529,0.0,0.006,41.339,0.0,63.584,0.0,0.0,1445.882,0,2077.340,1509.466,567.874,3755.400,0,83.0,0,2503.790,3738.43,1710.38,1564.035,0.525,4.13,63.405,529.920

## 8. Quality gates and export

These assertions fail loudly if a later source update changes the schema or time alignment.
The exported CSV has no missing values, no duplicate issue-time/horizon keys, and no target
timestamp at or before the forecast issue time.

In [9]:
natural_key = ["issue_timestamp_utc", "forecast_horizon_minutes"]
duplicate_rows = int(combined.duplicated(natural_key).sum())
missing_cells = int(combined.isna().sum().sum())

assert not combined.empty, "The merged dataset is empty. Check source coverage and schemas."
assert duplicate_rows == 0, "Duplicate issue-time/horizon keys found."
assert missing_cells == 0, "Missing values remain in the modelling table."
assert (combined["target_timestamp_utc"] > combined["issue_timestamp_utc"]).all()
assert (
    combined["target_timestamp_utc"] - combined["issue_timestamp_utc"]
    == pd.to_timedelta(combined["forecast_horizon_minutes"], unit="m")
).all()
assert (
    combined["dispatch_down_event"]
    == (combined["dispatch_down_mwh"] > 0).astype("int8")
).all()
assert (
    combined["recoverable_surplus_100mw_flex_mwh"]
    <= combined["dispatch_down_mwh"] + 1e-9
).all()

target_accounting_difference = (
    combined["dispatch_down_mwh"]
    - combined["curtailment_mwh"]
    - combined["constraint_mwh"]
).abs()
assert target_accounting_difference.max() < 0.05

combined.to_csv(OUTPUT_DATASET, index=False, date_format="%Y-%m-%dT%H:%M:%SZ")

event_summary = (
    combined.groupby("forecast_horizon_minutes")
    .agg(
        rows=("dispatch_down_event", "size"),
        event_rate=("dispatch_down_event", "mean"),
        dispatch_down_mwh_mean=("dispatch_down_mwh", "mean"),
        dispatch_down_mwh_p95=("dispatch_down_mwh", lambda x: x.quantile(0.95)),
    )
    .reset_index()
)

quality_report = {
    "dataset": OUTPUT_DATASET.relative_to(REPO_ROOT).as_posix(),
    "rows": int(len(combined)),
    "columns": int(combined.shape[1]),
    "issue_timestamp_min_utc": combined["issue_timestamp_utc"].min().isoformat(),
    "issue_timestamp_max_utc": combined["issue_timestamp_utc"].max().isoformat(),
    "forecast_horizons_minutes": list(FORECAST_HORIZONS_MINUTES),
    "duplicate_natural_keys": duplicate_rows,
    "missing_cells": missing_cells,
    "max_target_accounting_difference_mwh": float(target_accounting_difference.max()),
    "event_rate_by_horizon": {
        str(int(row.forecast_horizon_minutes)): float(row.event_rate)
        for row in event_summary.itertuples()
    },
    "assumptions": {
        "organiser_sample_timezone": "UTC; January overlap means Europe/Dublin offset is zero",
        "price_alignment": "Hourly IE_SEM prices carried forward once to 30-minute intervals",
        "short_gap_policy": "Time interpolation limited to two half-hour steps with flags",
        "flexible_load_scenario_mw": FLEXIBLE_LOAD_CAPACITY_MW,
    },
}
OUTPUT_QUALITY.write_text(json.dumps(quality_report, indent=2), encoding="utf-8")

print(f"Saved {OUTPUT_DATASET.relative_to(REPO_ROOT)}")
display(event_summary)

Saved data\processed\gridtoev_model_ready.csv


,forecast_horizon_minutes,rows,event_rate,dispatch_down_mwh_mean,dispatch_down_mwh_p95
0,30,1434,0.450488,41.438839,230.78745
1,60,1433,0.450105,41.416745,230.79480


## 9. Data dictionary

The dictionary is generated from the final table so it cannot silently omit engineered
columns. Prefixes identify provenance: `entsoe_` comes from the organiser samples,
`eirgrid_` comes from the system workbook, and dispatch-down targets come from EirGrid's
half-hourly label workbook.

In [10]:
def classify_column(column: str) -> tuple[str, str, str]:
    if column in {"issue_timestamp_utc", "target_timestamp_utc"}:
        return "Derived alignment", "identifier", "UTC timestamp"
    if column == "forecast_horizon_minutes":
        return "Derived alignment", "feature", "minutes"
    if column in target_columns:
        unit = "binary" if column == "dispatch_down_event" else "MWh per half-hour"
        return "EirGrid DD Half-Hourly 2026", "target", unit
    if column.startswith("entsoe_"):
        if "price" in column:
            unit = "EUR/MWh" if column.endswith("eur_mwh") else "binary"
        elif column.endswith("_flag"):
            unit = "binary"
        else:
            unit = "MW"
        return "Hack the Climate / ENTSO-E samples", "feature", unit
    if column.startswith("eirgrid_"):
        if column.endswith("_flag"):
            unit = "binary"
        elif column.endswith("_ratio") or "headroom_to_75pct" in column:
            unit = "ratio"
        else:
            unit = "MW"
        return "EirGrid System Data Qtr Hourly 2026", "feature", unit
    if column.startswith("dispatch_down_"):
        unit = "binary" if "event" in column else "MWh per half-hour"
        return "Derived from issue-time or lagged EirGrid labels", "feature", unit
    if column in {"weekday", "month"}:
        return "Derived calendar", "feature", "integer"
    if column.startswith("is_") or column.endswith("_flag"):
        return "Derived", "feature", "binary"
    if "price" in column:
        return "Derived from organiser price", "feature", "EUR/MWh"
    if any(token in column for token in ("wind", "load", "oversupply")):
        return "Derived rolling/lag feature", "feature", "MW"
    if "snsp" in column or "interaction" in column or column.endswith(("_sin", "_cos")):
        return "Derived", "feature", "ratio"
    return "Derived", "feature", "varies"


dictionary_rows = []
for column in combined.columns:
    source, role, unit = classify_column(column)
    dictionary_rows.append(
        {
            "column": column,
            "role": role,
            "unit": unit,
            "source": source,
            "dtype": str(combined[column].dtype),
        }
    )

data_dictionary = pd.DataFrame(dictionary_rows)
data_dictionary.to_csv(OUTPUT_DICTIONARY, index=False)
print(f"Saved {OUTPUT_DICTIONARY.relative_to(REPO_ROOT)}")
display(data_dictionary.head(15))

Saved data\processed\gridtoev_data_dictionary.csv


,column,role,unit,source,dtype
0,issue_timestamp_utc,identifier,UTC timestamp,Derived alignment,"datetime64[us, UTC]"
1,entsoe_fossil_gas_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64
2,entsoe_fossil_hard_coal_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64
3,entsoe_fossil_oil_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64
4,entsoe_fossil_peat_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64
5,entsoe_hydro_pumped_storage_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64
6,entsoe_hydro_run_of_river_and_poundage_generat...,feature,MW,Hack the Climate / ENTSO-E samples,float64
7,entsoe_other_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64
8,entsoe_solar_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64
9,entsoe_wind_onshore_generation_mw,feature,MW,Hack the Climate / ENTSO-E samples,float64


## Result

`gridtoev_model_ready.csv` is the single combined modelling dataset. For evaluation, split it
chronologically by `issue_timestamp_utc`; do not use a random train/test split. Keep both rows
for a multi-horizon model or filter `forecast_horizon_minutes` for separate 30- and 60-minute
models.